# Week 7: Buyer Intent Classification Deliverables

This notebook demonstrates the Week 7 deliverables:

- 200+ labeled user queries for intent classification
- Multi-class intent predictions with confidence scores
- Test-set evaluation with 80%+ accuracy target

In [27]:
from pathlib import Path
import sys
import importlib

# Ensure project root is importable when running from notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.intent_classifier as intent_classifier_module
importlib.reload(intent_classifier_module)

IntentClassifier = intent_classifier_module.IntentClassifier
build_default_labeled_dataset = intent_classifier_module.build_default_labeled_dataset

import pandas as pd

In [28]:
dataset = build_default_labeled_dataset()
df = pd.DataFrame(dataset, columns=["query", "intent"])

print("Total labeled queries:", len(df))
print("\nClass distribution:")
print(df["intent"].value_counts())

df.head(12)

Total labeled queries: 252

Class distribution:
intent
researching     84
browsing        84
ready_to_buy    84
Name: count, dtype: int64


,query,intent
0,is Austin a good market for first-time buyers,researching
1,what should I know before buying in Austin,researching
2,what is available this week in Phoenix,browsing
3,I am pre-approved and need homes below 500k in...,ready_to_buy
4,I am casually checking homes with updated kitchen,browsing
5,any nice homes around Irvine,browsing
6,just looking at duplexs with updated kitchen,browsing
7,which areas have lower taxes around San Diego,researching
8,which areas have lower taxes around Phoenix,researching
9,compare average prices for condos in San Diego,researching


In [29]:
classifier = IntentClassifier()

# Standard benchmark used for Week 7 deliverable
metrics = classifier.train_test_evaluate(dataset=dataset, test_size=0.2, split_strategy="stratified")
print("Stratified metrics:", metrics)
assert float(metrics["accuracy"]) >= 0.80, f"Accuracy below target: {float(metrics['accuracy']):.3f}"
print(f"✅ Accuracy target met (stratified split): {float(metrics['accuracy']):.3f}")

# Extra sanity check: harder split that holds out full template families
hard_metrics = classifier.train_test_evaluate(test_size=0.2, split_strategy="template_holdout")
print("Template-holdout metrics (harder generalization check):", hard_metrics)

Stratified metrics: {'accuracy': 1.0, 'train_size': 204.0, 'test_size': 48.0, 'split_strategy': 'stratified'}
✅ Accuracy target met (stratified split): 1.000
Template-holdout metrics (harder generalization check): {'accuracy': 0.9365079365079365, 'train_size': 189.0, 'test_size': 63.0, 'split_strategy': 'template_holdout'}


In [30]:
classifier = IntentClassifier()
queries = [q for q, _ in dataset]
labels = [y for _, y in dataset]
classifier.train(queries, labels)

sample_queries = [
    "just browsing homes in Irvine with a pool",
    "compare home prices and school quality in San Diego",
    "I am pre-approved and need a 3 bed under 900k in Austin",
    "find move-in ready homes and schedule tours this weekend",
]

rows = []
for query in sample_queries:
    pred = classifier.predict(query)
    rows.append(
        {
            "query": query,
            "predicted_intent": pred.intent,
            "confidence": round(pred.confidence, 4),
            "is_uncertain": pred.is_uncertain,
            "prob_browsing": round(pred.probabilities["browsing"], 4),
            "prob_researching": round(pred.probabilities["researching"], 4),
            "prob_ready_to_buy": round(pred.probabilities["ready_to_buy"], 4),
        }
    )

pd.DataFrame(rows)

,query,predicted_intent,confidence,is_uncertain,prob_browsing,prob_researching,prob_ready_to_buy
0,just browsing homes in Irvine with a pool,browsing,0.9996,False,0.9996,0.00,0.0004
1,compare home prices and school quality in San ...,researching,0.9800,False,0.0100,0.98,0.0100
2,I am pre-approved and need a 3 bed under 900k ...,ready_to_buy,0.9200,False,0.0400,0.04,0.9200
3,find move-in ready homes and schedule tours th...,ready_to_buy,0.9800,False,0.0100,0.01,0.9800


In [31]:
import csv

# Compute human-like holdout score so deliverables table is self-contained
human_holdout_path = PROJECT_ROOT / "data" / "processed" / "intent_human_like_eval.csv"
human_rows = []
with human_holdout_path.open("r", encoding="utf-8") as fh:
    reader = csv.DictReader(fh)
    for row in reader:
        human_rows.append((row["query"], row["intent"]))

holdout_classifier = IntentClassifier()
holdout_classifier.train([q for q, _ in dataset], [y for _, y in dataset])
correct = sum(1 for q, gold in human_rows if holdout_classifier.predict(q).intent == gold)
human_holdout_accuracy = correct / len(human_rows)

deliverables = pd.DataFrame(
    [
        {"deliverable": "Labeled dataset (200+)", "status": "PASS" if len(df) >= 200 else "FAIL", "value": len(df)},
        {
            "deliverable": "Test accuracy >= 80% (stratified)",
            "status": "PASS" if float(metrics["accuracy"]) >= 0.80 else "FAIL",
            "value": round(float(metrics["accuracy"]), 4),
        },
        {
            "deliverable": "Human-like holdout >= 80%",
            "status": "PASS" if human_holdout_accuracy >= 0.80 else "FAIL",
            "value": round(human_holdout_accuracy, 4),
        },
        {"deliverable": "Confidence scores included", "status": "PASS", "value": "intent_confidence + class probabilities"},
    ]
)

deliverables

,deliverable,status,value
0,Labeled dataset (200+),PASS,252
1,Test accuracy >= 80% (stratified),PASS,1.0
2,Human-like holdout >= 80%,PASS,0.8333
3,Confidence scores included,PASS,intent_confidence + class probabilities


In [32]:
import csv

human_holdout_path = PROJECT_ROOT / "data" / "processed" / "intent_human_like_eval.csv"
human_rows = []
with human_holdout_path.open("r", encoding="utf-8") as fh:
    reader = csv.DictReader(fh)
    for row in reader:
        human_rows.append((row["query"], row["intent"]))

classifier = IntentClassifier()
classifier.train([q for q, _ in dataset], [y for _, y in dataset])

correct = 0
human_eval_rows = []
for query, gold in human_rows:
    pred = classifier.predict(query)
    is_correct = pred.intent == gold
    correct += int(is_correct)
    human_eval_rows.append(
        {
            "query": query,
            "gold_intent": gold,
            "predicted_intent": pred.intent,
            "confidence": round(pred.confidence, 4),
            "correct": is_correct,
        }
    )

human_holdout_accuracy = correct / len(human_rows)
print(f"Human-like holdout accuracy: {human_holdout_accuracy:.4f} ({correct}/{len(human_rows)})")
assert human_holdout_accuracy >= 0.80, f"Human-like holdout accuracy below target: {human_holdout_accuracy:.3f}"

pd.DataFrame(human_eval_rows).head(12)

Human-like holdout accuracy: 0.8333 (50/60)


,query,gold_intent,predicted_intent,confidence,correct
0,I am just browsing what homes are available ne...,browsing,browsing,1.0000,True
1,Any interesting condos in Irvine right now?,browsing,browsing,0.9497,True
2,"Looking around for places with a backyard, no ...",browsing,browsing,1.0000,True
3,Show me options around Seattle with good natur...,browsing,browsing,0.9200,True
4,Just checking out what is on the market this m...,browsing,researching,0.9200,False
5,What are some homes to casually look at in San...,browsing,browsing,0.9200,True
6,I want to see a few neighborhoods before decid...,browsing,researching,0.9200,False
7,Can you pull up a mix of condos and townhomes ...,browsing,browsing,0.9200,True
8,"Not ready yet, just exploring homes near parks",browsing,browsing,1.0000,True
9,I am window shopping homes with modern kitchens,browsing,browsing,1.0000,True
